In [5]:
# [CELL 1 - UPDATED] CONFIGURATION & PARAMETERS
from pathlib import Path
from collections import Counter
import csv, json, shutil

LAST_NAME = "Ongjoco"
STUDENT_ID = "TUPM-26-1852"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_PREFIX = str((SEED_DIGIT % 4) + 2)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

# Ensure folders exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"
print(f"Target Records: {RECORDS_TO_PROCESS}, Target Status: {TARGET_STATUS_PREFIX}xx")

Target Records: 1024, Target Status: 4xx


In [6]:
# [CELL 2] PROCESS LOG STREAM
total_read = 0
target_count = 0
status_counts = Counter()
url_counts = Counter()
selected_records = []

if dataset_path.exists():
    with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip()
        for _ in range(RECORDS_TO_PROCESS):
            line = f.readline()
            if not line: break
            total_read += 1
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 4:
                ip, timestamp, url, status = parts[0], parts[1], parts[2], parts[3]
                status_counts[status] += 1
                if status.startswith(TARGET_STATUS_PREFIX):
                    target_count += 1
                    url_counts[url] += 1
                    selected_records.append({"ip": ip, "timestamp": timestamp, "url": url, "status": status})

print("Records Processed:", total_read)
print(f"Records in target class ({TARGET_STATUS_PREFIX}xx):", target_count)
print("Top 5 URLs:", url_counts.most_common(5))

Records Processed: 0
Records in target class (4xx): 0
Top 5 URLs: []


In [7]:
# [CELL 3] EXPORT CSV, JSON & BACKUP
csv_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"
json_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

with open(csv_out, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=["ip", "timestamp", "url", "status"])
    writer.writeheader()
    writer.writerows(selected_records)

summary = {
    "student": LAST_NAME, "student_id": STUDENT_ID,
    "records_processed": total_read, "target_class": TARGET_STATUS_PREFIX + "xx",
    "target_records": target_count, "status_breakdown": dict(status_counts),
    "top_5_urls": dict(url_counts.most_common(5))
}
with open(json_out, "w", encoding="utf-8") as j:
    json.dump(summary, j, indent=4)

backup_path = BACKUP_DIR / json_out.name
shutil.copy2(json_out, backup_path)
print("JSON Backup Exists:", backup_path.exists(), "| Size:", backup_path.stat().st_size, "bytes")

JSON Backup Exists: True | Size: 200 bytes
